In [2]:
# 1. Clone your collaboration repository
# Replace with your actual GitHub URL
!git clone https://github.com/nmorok/Teleconnections-ViT.git

# 2. Enter the repository directory
import os
os.chdir('Teleconnections-ViT')
# 3. Add the current directory to sys.path so 'import model' works
import sys
sys.path.append(os.getcwd())

# 4. Verify the files are present
print("Files in current directory:", os.listdir())

Cloning into 'Teleconnections-ViT'...
remote: Enumerating objects: 63, done.
remote: Counting objects: 100% (63/63), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 63 (delta 14), reused 55 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (63/63), 2.16 MiB | 14.06 MiB/s, done.
Resolving deltas: 100% (14/14), done.
Files in current directory: ['data', '.git', 'LICENSE', '.gitignore', 'requirements.txt', 'Pipeline.md', 'README.md', 'notebook', 'models']


In [ ]:
from google.colab import auth
auth.authenticate_user()


from google.colab import drive
import torch

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define your data path (Update this to your actual Drive folder)
# Usually formatted as: /content/drive/MyDrive/Folder_Name
DATA_PATH = '/content/drive/MyDrive/Teleconnection_ViT'

# 3. Quick verification check
if os.path.exists(DATA_PATH):
    print(f"✓ Data folder found at: {DATA_PATH}")
    print("Files available:", os.listdir(DATA_PATH))
else:
    print(f"✗ ERROR: Could not find folder at {DATA_PATH}. Check your Drive path.")

# 4. Hardware Check
# 1. Check if CUDA (GPU support) is available
print(f"Is CUDA available? {torch.cuda.is_available()}")

# 2. Get the name of the GPU assigned by Colab Pro
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_tensor_details_device(0)}")
else:
    print("WARNING: Using CPU. Check your Colab Runtime settings.")

KeyboardInterrupt: Interrupted by user

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from model import CrabTransformer
from losses import TweedieLoss
from data.data_helper import CrabDataset

In [ ]:
def train_model():
    # set hyperparameters:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    batch_size = 100
    epochs = 50
    learning_rate = 1e-4
    grid_size = 50
    patch_size = 5
    in_channels = 2
    embed_dim = 128
    num_heads = 8
    d_ff = 512
    num_layers = 6
    mask = False
    dropout = 0.1

    # prepare data
    train_ds = CrabDataset(
        spawner_path=os.path.join(DATA_PATH, "train_spawners.npy"), 
        recruit_path=os.path.join(DATA_PATH, "train_recruits.npy"),
        n_years=30
    )

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=False)

    val_ds = CrabDataset(
        spawner_path=os.path.join(DATA_PATH, "val_spawners.npy"), 
        recruit_path=os.path.join(DATA_PATH, "val_recruits.npy"),
        n_years=30
    )

    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    # initialize the model, loss, and optimizer
    model = CrabTransformer(grid_size, patch_size, in_channels, embed_dim, num_heads, num_layers, d_ff, mask, dropout).to(device)
    criterion = TweedieLoss(power=1.5)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    # training loop
    model.train()
    for epoch in range(epochs):
        total_train_loss = 0
        for batch_idx, (inputs, targets) in enumerate(train_loader):
            # 'data' is already the [Batch, 2, 10, 10] tensor
            # 'target' is the [Batch, 1, 10, 10] recruitment density
            inputs, targets = inputs.to(device), targets.to(device)

            # clear the gradients
            optimizer.zero_grad()
            # forward pass
            outputs = model(inputs)
            # calculate loss
            loss = criterion(outputs, targets)
            # backward pass (calculate gradients)
            loss.backward()
            # update weights
            optimizer.step()

            total_train_loss += loss.item()

        # --- VALIDATION PHASE ---
        model.eval() # Set model to evaluation mode
        total_val_loss = 0
        with torch.no_grad(): # Disable gradient calculation (saves memory/time)
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                total_val_loss += loss.item()

        avg_train = total_train_loss / len(train_loader)
        avg_val = total_val_loss / len(val_loader)
            
        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}")

In [ ]:
if __name__ == "__main__":
    train_model()